# Player Similarity Scouting — data pipeline

Builds the dataset used by the Streamlit app: pulls raw event data from StatsBomb Open Data (World Cup 2022 + Euro 2024), recomputes xG, progressive passes/carries and defensive stats from the raw coordinates, clusters players by playing style, and saves everything to `data/player_profiles.pkl`.

## 1. Setup

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path
from statsbombpy import sb

pd.set_option("display.max_columns", 50)

## 2. Competitions and matches

Free StatsBomb Open Data access only covers a handful of competitions (no full domestic seasons for the big leagues). Went with the 2022 World Cup and Euro 2024 since international tournaments still give a decent spread of players across the top leagues.

In [23]:
competitions = sb.competitions()

wc_row = competitions[
    (competitions["competition_name"] == "FIFA World Cup") & (competitions["season_name"] == "2022")
].iloc[0]
euro_row = competitions[
    (competitions["competition_name"] == "UEFA Euro") & (competitions["season_name"] == "2024")
].iloc[0]

matches_wc = sb.matches(competition_id=wc_row["competition_id"], season_id=wc_row["season_id"])
matches_euro = sb.matches(competition_id=euro_row["competition_id"], season_id=euro_row["season_id"])

all_matches = pd.concat([
    matches_wc.assign(competition="WC2022"),
    matches_euro.assign(competition="Euro2024"),
])

all_matches.shape

C:\Users\camca\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\camca\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


(115, 55)

## 3. Pulling all events

The slow part, 115 matches of event data. Cached to parquet so reruns don't hit the API again.

In [24]:
# quick look at one match before running the full loop, just to check the columns
# sample_match_id = all_matches.iloc[0]["match_id"]
# test_events = sb.events(match_id=sample_match_id)
# test_events["type"].value_counts()

events_path = Path("../data/events_wc2022_euro2024.parquet")  # notebook lives in notebook/, data/ is one level up

if events_path.exists():
    all_events = pd.read_parquet(events_path)
else:
    events_list = []
    for i, match_id in enumerate(all_matches["match_id"]):
        events = sb.events(match_id=match_id)
        events["match_id"] = match_id
        events_list.append(events)
        if (i + 1) % 10 == 0:
            print(f"{i + 1} / {len(all_matches)} matches done")

    all_events = pd.concat(events_list, ignore_index=True)
    events_path.parent.mkdir(exist_ok=True)
    all_events.to_parquet(events_path)

all_events.shape

(422561, 115)

## 4. Minutes played

Not given directly by StatsBomb, so it has to be rebuilt from `Starting XI` and `Substitution` events.

In [25]:
def get_match_minutes(events, match_id):
    match_events = events[events["match_id"] == match_id]
    duration = match_events["minute"].max()

    starting = match_events[match_events["type"] == "Starting XI"]
    minutes = {}

    for _, row in starting.iterrows():
        for player in row["tactics"]["lineup"]:
            minutes[player["player"]["id"]] = {"name": player["player"]["name"], "on": 0, "off": duration}

    subs = match_events[match_events["type"] == "Substitution"]
    for _, row in subs.iterrows():
        off_id = row["player_id"]
        sub_minute = row["minute"]
        if off_id in minutes:
            minutes[off_id]["off"] = sub_minute
        if pd.notna(row.get("substitution_replacement_id")):
            on_id = row["substitution_replacement_id"]
            minutes[on_id] = {"name": row["substitution_replacement"], "on": sub_minute, "off": duration}

    return pd.DataFrame([
        {"match_id": match_id, "player_id": pid, "player_name": v["name"], "minutes_played": v["off"] - v["on"]}
        for pid, v in minutes.items()
    ])

minutes_list = [get_match_minutes(all_events, mid) for mid in all_matches["match_id"]]
player_minutes = pd.concat(minutes_list, ignore_index=True)

player_minutes_total = (
    player_minutes.groupby(["player_id", "player_name"])["minutes_played"]
    .sum()
    .reset_index()
)

player_minutes_total.sort_values("minutes_played", ascending=False).head()

,player_id,player_name,minutes_played
57,3468.0,Jordan Pickford,1203
33,3244.0,John Stones,1182
101,3943.0,Declan Rice,1166
278,6704.0,Theo Bernard François Hernández,1150
436,10481.0,Aurélien Djani Tchouaméni,1150


## 5. Features recomputed from the raw events

This is the main point of using StatsBomb instead of a pre-aggregated source: xG is already included per shot, but progressive passes/carries and the defensive numbers are all built directly from each event's start and end coordinates.

In [26]:
def dist_to_goal(loc):
    if loc is None or (isinstance(loc, float) and np.isnan(loc)):
        return np.nan
    x, y = loc
    return np.sqrt((120 - x) ** 2 + (40 - y) ** 2)

def is_valid_loc(loc):
    # events reloaded from parquet come back as numpy arrays instead of plain python lists,
    # so an isinstance(loc, list) check here quietly breaks after a reload -> use this instead
    if loc is None:
        return False
    if isinstance(loc, float) and np.isnan(loc):
        return False
    return hasattr(loc, "__len__") and len(loc) == 2

def is_progressive(start, end):
    d_start = dist_to_goal(start)
    d_end = dist_to_goal(end)
    if np.isnan(d_start) or np.isnan(d_end):
        return False
    gain = d_start - d_end
    own_half = start[0] < 60
    opp_half = end[0] >= 60
    if own_half and not opp_half:
        return gain >= 30
    elif own_half and opp_half:
        return gain >= 15
    else:
        return gain >= 10

ev = all_events.copy()

shots = ev[ev["type"] == "Shot"].copy()
shots["is_goal"] = shots["shot_outcome"] == "Goal"
shot_stats = shots.groupby("player_id").agg(
    shots=("id", "count"),
    goals=("is_goal", "sum"),
    xg=("shot_statsbomb_xg", "sum"),
).reset_index()

passes = ev[ev["type"] == "Pass"].copy()
passes["is_progressive"] = passes.apply(
    lambda r: is_progressive(r["location"], r["pass_end_location"])
    if is_valid_loc(r["location"]) and is_valid_loc(r["pass_end_location"]) else False,
    axis=1,
)
pass_stats = passes.groupby("player_id").agg(
    passes_completed=("pass_outcome", lambda x: x.isna().sum()),
    progressive_passes=("is_progressive", "sum"),
).reset_index()

carries = ev[ev["type"] == "Carry"].copy()
carries["is_progressive"] = carries.apply(
    lambda r: is_progressive(r["location"], r["carry_end_location"])
    if is_valid_loc(r["location"]) and is_valid_loc(r["carry_end_location"]) else False,
    axis=1,
)
carry_stats = carries.groupby("player_id").agg(progressive_carries=("is_progressive", "sum")).reset_index()

defensive_types = ["Interception", "Clearance", "Block", "Ball Recovery", "Pressure"]
defense = ev[ev["type"].isin(defensive_types)]
defense_stats = defense.groupby(["player_id", "type"]).size().unstack(fill_value=0).reset_index()

player_profiles = (
    player_minutes_total
    .merge(shot_stats, on="player_id", how="left")
    .merge(pass_stats, on="player_id", how="left")
    .merge(carry_stats, on="player_id", how="left")
    .merge(defense_stats, on="player_id", how="left")
    .fillna(0)
)

player_profiles.head()

,player_id,player_name,minutes_played,shots,goals,xg,passes_completed,progressive_passes,progressive_carries,Ball Recovery,Block,Clearance,Interception,Pressure
0,2941.0,Ismaïla Sarr,363,10.0,1.0,1.848790,57.0,20.0,18.0,17.0,4.0,1.0,3.0,51.0
1,2954.0,Youri Tielemans,260,4.0,1.0,0.153319,152.0,34.0,4.0,9.0,10.0,0.0,4.0,34.0
2,2972.0,Marcus Thuram,460,13.0,0.0,1.294312,78.0,12.0,17.0,14.0,8.0,5.0,4.0,72.0
3,2988.0,Memphis Depay,780,28.0,2.0,3.132957,208.0,60.0,18.0,18.0,8.0,1.0,2.0,120.0
4,2989.0,Alexander Djiku,109,0.0,0.0,0.000000,23.0,6.0,0.0,3.0,1.0,10.0,4.0,5.0


## 6. Discipline: cards and fouls

Cards can come from two different event types in StatsBomb: one tied to a `Foul Committed`, or a standalone `Bad Behaviour` event (e.g. a second yellow for dissent, not linked to a specific foul). Missed the second one at first and undercounted red cards as a result.

In [27]:
bad_behaviour = ev[ev["type"] == "Bad Behaviour"].copy()
bad_behaviour["is_card"] = bad_behaviour["bad_behaviour_card"].notna()
bad_behaviour["is_red"] = bad_behaviour["bad_behaviour_card"].isin(["Red Card", "Second Yellow"])

fouls = ev[ev["type"] == "Foul Committed"].copy()
fouls["is_card"] = fouls["foul_committed_card"].notna()
fouls["is_red"] = fouls["foul_committed_card"].isin(["Red Card", "Second Yellow"])

cards_from_fouls = fouls.groupby("player_id").agg(
    fouls_committed=("id", "count"),
    cards_from_fouls=("is_card", "sum"),
    red_cards=("is_red", "sum"),
).reset_index()

cards_from_bad_behaviour = bad_behaviour.groupby("player_id").agg(
    cards_from_bad_behaviour=("is_card", "sum"),
    red_from_bad_behaviour=("is_red", "sum"),
).reset_index()

fouls_won_stats = ev[ev["type"] == "Foul Won"].groupby("player_id").size().reset_index(name="fouls_won")

player_profiles = (
    player_profiles
    .merge(cards_from_fouls, on="player_id", how="left")
    .merge(cards_from_bad_behaviour, on="player_id", how="left")
    .merge(fouls_won_stats, on="player_id", how="left")
    .fillna(0)
)

player_profiles["total_cards"] = player_profiles["cards_from_fouls"] + player_profiles["cards_from_bad_behaviour"]
player_profiles["red_cards"] = player_profiles["red_cards"] + player_profiles["red_from_bad_behaviour"]
player_profiles["yellow_cards"] = player_profiles["total_cards"] - player_profiles["red_cards"]

# Dumfries got a second yellow vs Argentina in the WC -> good sanity check that this is counted right
player_profiles[player_profiles["player_name"].str.contains("Dumfries")][["player_name", "total_cards", "yellow_cards", "red_cards"]]

,player_name,total_cards,yellow_cards,red_cards
351,Denzel Dumfries,4.0,3.0,1.0


## 7. Minutes filter and per-90 normalization

Dropping anyone under 450 minutes across both tournaments, so a few cameo appearances don't produce misleading per-90 numbers.

In [28]:
MIN_MINUTES = 450

df = player_profiles[player_profiles["minutes_played"] >= MIN_MINUTES].copy()
df["nineties"] = df["minutes_played"] / 90

per90_source_cols = ["shots", "goals", "xg", "passes_completed", "progressive_passes",
                      "progressive_carries", "Ball Recovery", "Block", "Clearance", "Interception",
                      "Pressure", "fouls_committed"]

for col in per90_source_cols:
    df[f"{col}_p90"] = df[col] / df["nineties"]

df["cards_p90"] = df["total_cards"] / df["nineties"]

df.shape

(130, 36)

## 8. Positions, teams and competition

In [29]:
lineups_list = []
for match_id in all_matches["match_id"]:
    lineup = sb.lineups(match_id=match_id)
    for team_name, team_df in lineup.items():
        team_df = team_df.copy()
        team_df["team"] = team_name
        keep_cols = ["player_id", "player_name", "positions", "team"]
        if "player_nickname" in team_df.columns:
            keep_cols.append("player_nickname")
        lineups_list.append(team_df[keep_cols])

all_lineups = pd.concat(lineups_list, ignore_index=True)
if "player_nickname" not in all_lineups.columns:
    all_lineups["player_nickname"] = None

def main_position(positions):
    if not positions:
        return None
    return positions[0]["position"]

all_lineups["position"] = all_lineups["positions"].apply(main_position)
all_lineups_valid = all_lineups.dropna(subset=["position"])

def most_common_position(x):
    counts = x.value_counts()
    return counts.idxmax() if len(counts) > 0 else None

player_positions = (
    all_lineups_valid.groupby("player_id")["position"]
    .agg(most_common_position)
    .reset_index()
)

all_teams = all_lineups.drop_duplicates(subset="player_id")[["player_id", "team"]]

# StatsBomb's player_name is the full official name (middle names included), which Wikidata search
# often misses -> keep the nickname around too, it's usually the common name and matches much better
player_nicknames = all_lineups.dropna(subset=["player_nickname"]).drop_duplicates(subset="player_id")[["player_id", "player_nickname"]]

df = df.merge(player_positions, on="player_id", how="left")
df = df.merge(all_teams, on="player_id", how="left")
df = df.merge(player_nicknames, on="player_id", how="left")

def position_group(pos):
    if pos is None or pd.isna(pos):
        return "Unknown"
    pos = str(pos)
    if "Goalkeeper" in pos:
        return "Goalkeeper"
    if "Back" in pos:
        return "Defender"
    if "Midfield" in pos:
        return "Midfielder"
    if "Wing" in pos or "Forward" in pos:
        return "Attacker"
    return "Other"

df["position_group"] = df["position"].apply(position_group)

competition_map = all_matches.set_index("match_id")["competition"].to_dict()
player_competition = (
    player_minutes[["match_id", "player_id"]]
    .assign(competition=lambda x: x["match_id"].map(competition_map))
    .groupby("player_id")["competition"]
    .agg(lambda x: " + ".join(sorted(set(x))))
    .reset_index()
)
df = df.merge(player_competition, on="player_id", how="left")

df[["player_name", "position", "position_group", "team", "competition"]].head()

C:\Users\camca\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\camca\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\camca\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
C:\Users\camca\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
 

,player_name,position,position_group,team,competition
0,Marcus Thuram,Center Forward,Attacker,France,Euro2024 + WC2022
1,Memphis Depay,Center Forward,Attacker,Netherlands,Euro2024 + WC2022
2,Kylian Mbappé Lottin,Left Wing,Attacker,France,Euro2024 + WC2022
3,Adrien Rabiot,Left Defensive Midfield,Midfielder,France,Euro2024 + WC2022
4,Christian Dannemann Eriksen,Left Center Midfield,Midfielder,Denmark,Euro2024 + WC2022


## 9. Scaling, PCA and clustering

No position constraint on the clustering on purpose, letting the model find playing-style groups on its own was the whole point (modern midfielders especially tend to blur position lines).

In [30]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

radar_cols = ["shots_p90", "goals_p90", "xg_p90", "passes_completed_p90", "progressive_passes_p90",
              "progressive_carries_p90", "Ball Recovery_p90", "Interception_p90", "Pressure_p90"]

X = df[radar_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(X_scaled)
print("components kept:", X_pca.shape[1])

for k in range(3, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    print(k, round(silhouette_score(X_pca, labels), 3))

components kept: 6
3 0.264
4 0.274
5 0.26
6 0.265
7 0.252
8 0.224
9 0.225
10 0.222


k=4 gave almost the same silhouette score as k=3, but with enough room to actually separate different attacking/defensive styles instead of lumping everything together.

In [31]:
k_final = 4
km_final = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(X_pca)

df["cluster"].value_counts().sort_index()

cluster
0    14
1    15
2    52
3    49
Name: count, dtype: int64

In [32]:
# 2D projection, just for the style map in the app
pca_2d = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
df["pca_x"] = pca_2d[:, 0]
df["pca_y"] = pca_2d[:, 1]

## 10. Similarity matrix

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(X_pca)
similarity_matrix.shape

(130, 130)

## 11. Percentiles by position group

In [34]:
percentile_cols = [c for c in df.columns if c.endswith("_p90")]

for col in percentile_cols:
    df[f"{col}_pct"] = df.groupby("position_group")[col].rank(pct=True) * 100

df[["player_name", "position_group", "xg_p90_pct"]].head()

,player_name,position_group,xg_p90_pct
0,Marcus Thuram,Attacker,44.827586
1,Memphis Depay,Attacker,65.517241
2,Kylian Mbappé Lottin,Attacker,86.206897
3,Adrien Rabiot,Midfielder,47.058824
4,Christian Dannemann Eriksen,Midfielder,67.647059


## 12. Player age and photo via Wikidata

No age, photo or market value in StatsBomb. Transfermarkt would be the natural source for market value but it blocks scraping, so that's left out entirely (noted as a limitation in the app). Age and a profile photo are pulled from Wikidata instead, which is free and doesn't have the same anti-bot issues FBref ran into.

First attempt was one REST call per player (search + claims), roughly 300+ requests for the whole sample, and that got rate-limited hard by Wikidata halfway through. Switched to the SPARQL query service instead: it can match a whole batch of names against Wikidata in a single query, so the entire sample fits in a handful of requests instead of hundreds.

In [35]:
import requests
import time

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
WIKI_USER_AGENT = "DataScienceFootProject/1.0 (student project, Polytech Nice)"
COMMONS_FILEPATH = "https://commons.wikimedia.org/wiki/Special:FilePath/"
FOOTBALLER_QID = "Q937857"  # "association football player" on Wikidata

# a handful of names don't resolve well through automated matching (compound Spanish/Portuguese first
# names especially) -> patched by hand after checking them individually
MANUAL_NAME_OVERRIDES = {
    "Lionel Andres Messi Cuccittini": "Lionel Messi",
    "Alexis Mac Allister": "Alexis Mac Allister",  # StatsBomb's nickname has no space ("MacAllister"), Wikidata's label does
}

def sparql_escape(name):
    return name.replace(chr(92), chr(92) * 2).replace('"', chr(92) + '"')

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

def run_sparql_batch(names):
    # tagging the values with @en lets the query engine do a direct indexed literal lookup instead of
    # scanning every label in every language for every entity on Wikidata -> first version timed out
    # and came back with a truncated, broken JSON body because of exactly that
    values = " ".join(f'"{sparql_escape(n)}"@en' for n in names)
    query = f"""
    SELECT DISTINCT ?name ?person ?dob ?image WHERE {{
      {{
        VALUES ?name {{ {values} }}
        ?person rdfs:label ?name .
      }} UNION {{
        VALUES ?name {{ {values} }}
        ?person skos:altLabel ?name .
      }}
      ?person wdt:P106 wd:{FOOTBALLER_QID} .
      OPTIONAL {{ ?person wdt:P569 ?dob . }}
      OPTIONAL {{ ?person wdt:P18 ?image . }}
    }}
    """
    resp = requests.post(
        SPARQL_ENDPOINT, data={"query": query},
        headers={"User-Agent": WIKI_USER_AGENT, "Accept": "application/sparql-results+json"},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["results"]["bindings"]

def parse_bindings(bindings):
    # a name can match more than one person (homonyms) -> just keep the first hit for each
    out = {}
    for b in bindings:
        name = b["name"]["value"]
        if name in out:
            continue
        dob = b["dob"]["value"][:10] if "dob" in b else None
        image_url = COMMONS_FILEPATH + b["image"]["value"].rsplit("/", 1)[-1] if "image" in b else None
        out[name] = (dob, image_url)
    return out

players = df[["player_name", "player_nickname"]].drop_duplicates(subset="player_name").copy()
players["search_name"] = players.apply(
    lambda r: MANUAL_NAME_OVERRIDES.get(r["player_name"])
    or (r["player_nickname"] if pd.notna(r["player_nickname"]) else r["player_name"]),
    axis=1,
)

search_to_players = {}
for _, r in players.iterrows():
    search_to_players.setdefault(r["search_name"], []).append(r["player_name"])

wiki_info = {}
for batch in chunked(list(search_to_players.keys()), 40):
    results = parse_bindings(run_sparql_batch(batch))
    for search_name in batch:
        for player_name in search_to_players[search_name]:
            wiki_info[player_name] = results.get(search_name, (None, None))
    time.sleep(1.0)

# second pass: for anyone still unmatched, retry with just the first two words of the full name
still_missing = [pn for pn, v in wiki_info.items() if v == (None, None)]
fallback_map = {(" ".join(pn.split()[:2]) if len(pn.split()) > 2 else pn): pn for pn in still_missing}

if fallback_map:
    fallback_names = list(fallback_map.keys())
    for batch in chunked(fallback_names, 40):
        results = parse_bindings(run_sparql_batch(batch))
        for fname in batch:
            if fname in results:
                wiki_info[fallback_map[fname]] = results[fname]
        time.sleep(1.0)

df["birthdate"] = pd.to_datetime(df["player_name"].map(lambda n: wiki_info.get(n, (None, None))[0]), errors="coerce")
today = pd.Timestamp.today()
df["age"] = ((today - df["birthdate"]).dt.days / 365.25).round(1)
df["image_url"] = df["player_name"].map(lambda n: wiki_info.get(n, (None, None))[1])

found = df["image_url"].notna().sum()
print(f"{found} / {len(df)} players got a photo")
df[["player_name", "player_nickname", "age", "image_url"]].head()

112 / 130 players got a photo


,player_name,player_nickname,age,image_url
0,Marcus Thuram,NaN,29.2,https://commons.wikimedia.org/wiki/Special:Fil...
1,Memphis Depay,NaN,32.7,https://commons.wikimedia.org/wiki/Special:Fil...
2,Kylian Mbappé Lottin,Kylian Mbappé,27.8,None
3,Adrien Rabiot,NaN,31.6,https://commons.wikimedia.org/wiki/Special:Fil...
4,Christian Dannemann Eriksen,Christian Eriksen,34.7,https://commons.wikimedia.org/wiki/Special:Fil...


### 12b. Cleanup pass for the stragglers

The SPARQL exact-label match misses a few well-known players for reasons that aren't worth chasing further (encoding quirks, label vs alias mismatches). There's only a couple dozen left at this point, so it's cheap to fall back to the slower but more forgiving fuzzy search (`wbsearchentities`) just for those, one request at a time.

In [36]:
WIKI_SESSION = requests.Session()
WIKI_SESSION.headers.update({"User-Agent": WIKI_USER_AGENT})

def wiki_api_call(params, retries=3):
    last_error = None
    for attempt in range(retries + 1):
        try:
            resp = WIKI_SESSION.get("https://www.wikidata.org/w/api.php", params=params, timeout=8)
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 15))
                print(f"rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            last_error = e
            time.sleep(2 * (attempt + 1))
    raise last_error

def fuzzy_lookup(name):
    search = wiki_api_call({"action": "wbsearchentities", "search": name, "language": "en", "format": "json", "type": "item"})
    hits = search.get("search") or []
    for hit in hits[:5]:
        entity_id = hit["id"]
        claims = wiki_api_call({"action": "wbgetclaims", "entity": entity_id, "format": "json"})["claims"]
        occupations = [c["mainsnak"]["datavalue"]["value"]["id"] for c in claims.get("P106", [])
                       if "datavalue" in c["mainsnak"]]
        if FOOTBALLER_QID not in occupations:
            continue  # same name, wrong person (homonym) -> try the next search hit instead
        dob = claims["P569"][0]["mainsnak"]["datavalue"]["value"]["time"][1:11] if "P569" in claims else None
        image_url = None
        if "P18" in claims:
            filename = claims["P18"][0]["mainsnak"]["datavalue"]["value"]
            image_url = COMMONS_FILEPATH + filename.replace(" ", "_")
        return dob, image_url
    return None, None

# a "straggler" here means missing a photo specifically, not missing a Wikidata match altogether:
# plenty of players already have a birthdate from the SPARQL pass but no P18 image on their entry
stragglers = [pn for pn, v in wiki_info.items() if v[1] is None]
print(f"{len(stragglers)} players left to try individually")

for player_name in stragglers:
    nickname = players.loc[players["player_name"] == player_name, "search_name"].iloc[0]
    try:
        dob, image_url = fuzzy_lookup(nickname)
        existing_dob, _ = wiki_info.get(player_name, (None, None))
        wiki_info[player_name] = (existing_dob or dob, image_url)
    except Exception as e:
        # don't let one stubborn lookup kill the rest of the batch -> just flag it and move on,
        # this cell is safe to rerun afterwards since it only retries names still missing a photo
        print(player_name, "-> gave up:", repr(e))
    time.sleep(1.0)

df["birthdate"] = pd.to_datetime(df["player_name"].map(lambda n: wiki_info.get(n, (None, None))[0]), errors="coerce")
df["age"] = ((today - df["birthdate"]).dt.days / 365.25).round(1)
df["image_url"] = df["player_name"].map(lambda n: wiki_info.get(n, (None, None))[1])

found = df["image_url"].notna().sum()
print(f"{found} / {len(df)} players got a photo (after cleanup pass)")

18 players left to try individually
rate limited, waiting 10s...
rate limited, waiting 52s...
rate limited, waiting 52s...
129 / 130 players got a photo (after cleanup pass)


In [37]:
missing_photo = df[df["image_url"].isna()]["player_name"].tolist()
missing_photo

['Michel Aebischer']

Just one player left without a photo: Michel Aebischer. Checked this one by hand directly on Wikidata (no `P18` claim on his entry at all) and on Wikimedia Commons (no usable free-licensed file under his name either), so it's not a search or name-matching issue, there simply isn't a freely-licensed photo of him available anywhere right now. The app shows his initials instead rather than forcing in a stock photo from somewhere else.

## 13. Save

In [38]:
import pickle

with open("../data/player_profiles.pkl", "wb") as f:
    pickle.dump({
        "df": df,
        "similarity_matrix": similarity_matrix,
        "radar_cols": radar_cols,
    }, f)

print("saved:", df.shape)

saved: (130, 60)
